## Cardiac Segmentation — 5-Fold Cross-Validation Evaluation

Full evaluation of trained models with Dice, IoU, Hausdorff Distance, MAD, confusion matrix, per-view/phase analysis, and qualitative visualizations.

## Setup
This notebook is designed for **Google Colab** with trained checkpoints stored in Google Drive.

**Before running**, update these paths:
- `zip_path` in Cell 0: path to your CAMUS `database_nifti.zip`
- `FOLD_CKPTS` in Cell 2: paths to your 5 trained checkpoint files
- `SAVE_DIR` in Cell 2: path where figures and results will be saved

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 0: Environment Setup — Install, Mount Drive, Extract Data
# ══════════════════════════════════════════════════════════════

# Install dependencies
!pip install segmentation-models-pytorch ttach albumentations pytorch-lightning -q

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Extract CAMUS dataset
import os, zipfile

zip_path = "/content/drive/MyDrive/CAMUS/database_nifti.zip"

print("Unzipping...")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall("/content")
print("Done!")

# Verify
patients = [f for f in os.listdir("/content/database_nifti") if f.startswith("patient")]
print(f"Number of patients: {len(patients)}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 1: Imports, Constants, Model & Data Utilities
# ══════════════════════════════════════════════════════════════

import os, glob, json, random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import ndimage
from scipy.spatial.distance import directed_hausdorff
from scipy.ndimage import binary_erosion
from scipy.spatial import cKDTree
from sklearn.metrics import confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F

import pytorch_lightning as pl
import segmentation_models_pytorch as smp
import ttach as tta
import nibabel as nib
import cv2

# ── Constants ──
SEED = 42
IMG_SIZE = 256
NUM_CLASSES = 4
N_FOLDS = 5
DATA_DIR = "/content/database_nifti"

CLASS_NAMES = ['Background', 'LV Cavity', 'Myocardium', 'Left Atrium']

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Model ──

class DeepSupSMPUNet(nn.Module):
    """U-Net with pretrained ResNet34, SCSE attention, and deep supervision."""

    def __init__(self, encoder_name='resnet34', encoder_weights='imagenet',
                 in_channels=1, num_classes=4):
        super().__init__()

        self.base = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=in_channels,
            classes=num_classes,
            decoder_attention_type='scse',
            decoder_channels=(256, 128, 64, 32, 16),
        )

        self._intermediate = {}
        for i, block in enumerate(self.base.decoder.blocks[:-1]):
            block.register_forward_hook(self._make_hook(i))

        ds_channels = [256, 128, 64, 32]
        self.aux_heads = nn.ModuleList([
            nn.Conv2d(ch, num_classes, kernel_size=1) for ch in ds_channels
        ])

        for head in self.aux_heads:
            nn.init.kaiming_normal_(head.weight, mode='fan_out', nonlinearity='relu')
            nn.init.zeros_(head.bias)

    def _make_hook(self, idx):
        def hook(module, input, output):
            self._intermediate[idx] = output
        return hook

    def forward(self, x):
        H, W = x.shape[2:]
        main_out = self.base(x)

        if self.training:
            aux_outputs = []
            for i, head in enumerate(self.aux_heads):
                feat = self._intermediate[i]
                aux = head(feat)
                aux = F.interpolate(aux, size=(H, W), mode='bilinear', align_corners=False)
                aux_outputs.append(aux)
            return main_out, aux_outputs

        return main_out


# ── Data Discovery + Fold Splits ──

def discover_patient_samples(data_dir):
    """Scan dataset and return {patient_id: [(img_path, mask_path), ...]}."""
    patient_dirs = sorted(glob.glob(os.path.join(data_dir, 'patient*')))
    print(f"Total patient folders found: {len(patient_dirs)}")

    patient_samples = {}
    for pdir in patient_dirs:
        patient_id = os.path.basename(pdir)
        samples = []
        gt_files = sorted(glob.glob(os.path.join(pdir, '*_gt.nii.gz')))
        for gt_path in gt_files:
            img_path = gt_path.replace('_gt.nii.gz', '.nii.gz')
            if os.path.exists(img_path):
                samples.append((img_path, gt_path))
        if samples:
            patient_samples[patient_id] = samples

    total = sum(len(s) for s in patient_samples.values())
    print(f"Total samples: {total} across {len(patient_samples)} patients "
          f"({total / len(patient_samples):.1f} samples/patient)")
    return patient_samples


def get_fold_splits(patient_samples, n_folds=5, seed=42):
    """Patient-level k-fold split."""
    patient_ids = sorted(patient_samples.keys())
    rng = np.random.RandomState(seed)
    rng.shuffle(patient_ids)

    fold_size = len(patient_ids) // n_folds
    folds = []

    for fold_idx in range(n_folds):
        val_start = fold_idx * fold_size
        val_end = val_start + fold_size if fold_idx < n_folds - 1 else len(patient_ids)

        val_pids = patient_ids[val_start:val_end]
        train_pids = patient_ids[:val_start] + patient_ids[val_end:]

        train_samples = [s for pid in train_pids for s in patient_samples[pid]]
        val_samples = [s for pid in val_pids for s in patient_samples[pid]]

        folds.append((train_samples, val_samples))
        print(f"  Fold {fold_idx}: {len(train_pids)} train patients ({len(train_samples)} samples) | "
              f"{len(val_pids)} val patients ({len(val_samples)} samples)")

    return folds


print("✅ All imports, model, and utilities loaded.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 2: 5-Fold Cross-Validation — Full Evaluation
# ══════════════════════════════════════════════════════════════

# ── Config ──
FOLD_CKPTS = {
    0: "/content/drive/MyDrive/CAMUS/results/final/v3cv_fold0/v3cv-fold0-epoch=436-val_dice=0.9148.ckpt",
    1: "/content/drive/MyDrive/CAMUS/results/final/v3cv_fold1/v3cv-fold1-epoch=389-val_dice=0.9109.ckpt",
    2: "/content/drive/MyDrive/CAMUS/results/final/v3cv_fold2/v3cv-fold2-epoch=509-val_dice=0.9121.ckpt",
    3: "/content/drive/MyDrive/CAMUS/results/final/v3cv_fold3/v3cv-fold3-epoch=396-val_dice=0.9141.ckpt",
    4: "/content/drive/MyDrive/CAMUS/results/final/v3cv_fold4/v3cv-fold4-epoch=183-val_dice=0.9106.ckpt",
}
SAVE_DIR = "/content/drive/MyDrive/CAMUS/results/final/"

SEG_COLORS = {
    0: [0, 0, 0],
    1: [0, 255, 255],
    2: [255, 255, 0],
    3: [139, 0, 0],
}

# ── Helper functions ──

def load_sample(img_path, mask_path, img_size=256):
    img = nib.load(img_path).get_fdata().astype(np.float32)
    mask = nib.load(mask_path).get_fdata().astype(np.float32)
    if img.ndim == 3:
        img = img[:, :, 0]
        mask = mask[:, :, 0]
    img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
    mask = np.round(mask).astype(np.int64)
    mask = np.clip(mask, 0, 3)
    if img.max() > img.min():
        img = (img - img.min()) / (img.max() - img.min())
    img_t = torch.from_numpy(img).unsqueeze(0).float()
    mask_t = torch.from_numpy(mask).long()
    return img_t, mask_t, img, mask

def parse_filename(filepath):
    fname = os.path.basename(filepath)
    view = '2CH' if '2CH' in fname else '4CH' if '4CH' in fname else 'unknown'
    if 'half_sequence' in fname:
        phase = 'seq'
    elif '_ED' in fname:
        phase = 'ED'
    elif '_ES' in fname:
        phase = 'ES'
    else:
        phase = 'unknown'
    return view, phase

def dice_score(pred_c, gt_c):
    inter = (pred_c * gt_c).sum()
    union = pred_c.sum() + gt_c.sum()
    return (2.0 * inter / union) if union > 0 else np.nan

def iou_score(pred_c, gt_c):
    inter = (pred_c * gt_c).sum()
    union = pred_c.sum() + gt_c.sum() - inter
    return (inter / union) if union > 0 else np.nan

def hausdorff_dist(pred_c, gt_c):
    pred_pts = np.argwhere(pred_c)
    gt_pts = np.argwhere(gt_c)
    if len(pred_pts) == 0 or len(gt_pts) == 0:
        return np.nan
    return max(directed_hausdorff(pred_pts, gt_pts)[0],
               directed_hausdorff(gt_pts, pred_pts)[0])

def mean_absolute_dist(pred_c, gt_c):
    pred_contour = pred_c.astype(bool) & ~binary_erosion(pred_c.astype(bool))
    gt_contour = gt_c.astype(bool) & ~binary_erosion(gt_c.astype(bool))
    pred_pts = np.argwhere(pred_contour)
    gt_pts = np.argwhere(gt_contour)
    if len(pred_pts) == 0 or len(gt_pts) == 0:
        return np.nan
    tree_gt = cKDTree(gt_pts)
    tree_pred = cKDTree(pred_pts)
    d_pred_to_gt, _ = tree_gt.query(pred_pts)
    d_gt_to_pred, _ = tree_pred.query(gt_pts)
    return (d_pred_to_gt.mean() + d_gt_to_pred.mean()) / 2.0

def post_process(pred_mask):
    result = np.zeros_like(pred_mask)
    for c in range(1, NUM_CLASSES):
        binary = (pred_mask == c).astype(np.uint8)
        if binary.sum() == 0:
            continue
        labeled, num_features = ndimage.label(binary)
        if num_features == 0:
            continue
        sizes = np.bincount(labeled.flat)[1:]
        largest = sizes.argmax() + 1
        result[labeled == largest] = c
    return result

def mask_to_rgb(mask):
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for c, color in SEG_COLORS.items():
        rgb[mask == c] = color
    return rgb

def compute_stats(results, metric_prefix, classes=range(1, NUM_CLASSES)):
    per_class = {}
    for c in classes:
        key = f'{metric_prefix}_{c}'
        vals = [r[key] for r in results if key in r and not np.isnan(r[key])]
        per_class[c] = np.mean(vals) if vals else np.nan
    fg_mean = np.nanmean([per_class[c] for c in classes])
    return per_class, fg_mean


# ══════════════════════════════════════════════════════════════
# 1. EVALUATE ALL FOLDS
# ══════════════════════════════════════════════════════════════

patient_samples = discover_patient_samples(DATA_DIR)
folds = get_fold_splits(patient_samples, n_folds=N_FOLDS, seed=SEED)

all_results = []

for fold in range(N_FOLDS):
    print(f"\n{'='*60}")
    print(f"  FOLD {fold + 1} / {N_FOLDS}")
    print(f"{'='*60}")

    ckpt_path = FOLD_CKPTS[fold]
    print(f"  Loading: {os.path.basename(ckpt_path)}")

    ckpt = torch.load(ckpt_path, map_location='cuda')
    model = DeepSupSMPUNet(
        encoder_name='resnet34',
        encoder_weights=None,
        in_channels=1,
        num_classes=NUM_CLASSES,
    )
    state_dict = {k.replace('model.', '', 1): v
                  for k, v in ckpt['state_dict'].items()
                  if k.startswith('model.')}
    model.load_state_dict(state_dict)
    model.eval().cuda()

    tta_model = tta.SegmentationTTAWrapper(
        model, tta.Compose([tta.HorizontalFlip()]), merge_mode='mean')
    tta_model.eval().cuda()

    _, val_samples = folds[fold]
    n_val = len(val_samples)
    print(f"  Evaluating {n_val} validation samples...")

    fold_count = 0
    with torch.no_grad(), torch.amp.autocast('cuda'):
        for img_path, mask_path in val_samples:
            view, phase = parse_filename(img_path)
            img_t, mask_t, img_np, mask_np = load_sample(img_path, mask_path, IMG_SIZE)

            x = img_t.unsqueeze(0).cuda()

            raw_pred = model(x).argmax(dim=1).cpu().numpy()[0]
            tta_pred = tta_model(x).argmax(dim=1).cpu().numpy()[0]
            pp_pred = post_process(tta_pred)

            sample_result = {
                'fold': fold,
                'img_path': img_path,
                'view': view,
                'phase': phase,
                'img_np': img_np,
                'mask_np': mask_np,
                'pred_np': pp_pred,
            }

            for c in range(NUM_CLASSES):
                gt_c = (mask_np == c).astype(float)
                pred_raw_c = (raw_pred == c).astype(float)
                pred_tta_c = (pp_pred == c).astype(float)

                if gt_c.sum() == 0 and c > 0:
                    continue

                sample_result[f'dice_raw_{c}'] = dice_score(pred_raw_c, gt_c)
                sample_result[f'dice_tta_{c}'] = dice_score(pred_tta_c, gt_c)
                sample_result[f'iou_tta_{c}'] = iou_score(pred_tta_c, gt_c)

                if c > 0:
                    sample_result[f'hd_tta_{c}'] = hausdorff_dist(pred_tta_c, gt_c)
                    sample_result[f'mad_tta_{c}'] = mean_absolute_dist(pred_tta_c, gt_c)

            all_results.append(sample_result)
            fold_count += 1

    print(f"  ✅ Fold {fold} done — {fold_count} samples evaluated")

    del model, tta_model
    torch.cuda.empty_cache()


# ══════════════════════════════════════════════════════════════
# 2. AGGREGATE AND REPORT
# ══════════════════════════════════════════════════════════════

print(f"\n\n{'='*70}")
print("5-FOLD CROSS-VALIDATION — OVERALL RESULTS")
print(f"{'='*70}")

print(f"\n{'─'*70}")
print(f"  {'Fold':<6} {'Dice (raw)':>12} {'Dice (TTA)':>12} {'IoU':>10} {'HD (px)':>10} {'MAD (px)':>10}")
print(f"{'─'*70}")

fold_stats = {}
for fold in range(N_FOLDS):
    fold_res = [r for r in all_results if r['fold'] == fold]
    _, dice_raw_fg = compute_stats(fold_res, 'dice_raw')
    _, dice_tta_fg = compute_stats(fold_res, 'dice_tta')
    _, iou_fg = compute_stats(fold_res, 'iou_tta')
    _, hd_fg = compute_stats(fold_res, 'hd_tta')
    _, mad_fg = compute_stats(fold_res, 'mad_tta')
    fold_stats[fold] = {
        'dice_raw': dice_raw_fg, 'dice_tta': dice_tta_fg,
        'iou': iou_fg, 'hd': hd_fg, 'mad': mad_fg,
    }
    print(f"  {fold:<6} {dice_raw_fg:>12.4f} {dice_tta_fg:>12.4f} {iou_fg:>10.4f} {hd_fg:>10.2f} {mad_fg:>10.2f}")

print(f"{'─'*70}")
for metric, label in [('dice_raw', 'Dice (raw)'), ('dice_tta', 'Dice (TTA)'),
                       ('iou', 'IoU'), ('hd', 'HD (px)'), ('mad', 'MAD (px)')]:
    vals = [fold_stats[f][metric] for f in range(N_FOLDS)]
    fmt = '.4f' if metric not in ('hd', 'mad') else '.2f'
    print(f"  {'Mean±Std':<6} {label + ':':<14} {np.mean(vals):{fmt}} ± {np.std(vals):{fmt}}")
print(f"{'='*70}")

print(f"\n{'─'*70}")
print("PER-CLASS METRICS (TTA+PP, mean ± std across 5 folds)")
print(f"{'─'*70}")
print(f"  {'Class':<16} {'Dice':>12} {'IoU':>12} {'HD (px)':>12} {'MAD (px)':>12}")
print(f"{'─'*70}")

for c in range(NUM_CLASSES):
    dice_vals, iou_vals, hd_vals, mad_vals = [], [], [], []
    for fold in range(N_FOLDS):
        fold_res = [r for r in all_results if r['fold'] == fold]
        d = [r[f'dice_tta_{c}'] for r in fold_res if f'dice_tta_{c}' in r and not np.isnan(r[f'dice_tta_{c}'])]
        dice_vals.append(np.mean(d) if d else np.nan)
        i = [r[f'iou_tta_{c}'] for r in fold_res if f'iou_tta_{c}' in r and not np.isnan(r[f'iou_tta_{c}'])]
        iou_vals.append(np.mean(i) if i else np.nan)
        if c > 0:
            h = [r[f'hd_tta_{c}'] for r in fold_res if f'hd_tta_{c}' in r and not np.isnan(r[f'hd_tta_{c}'])]
            hd_vals.append(np.mean(h) if h else np.nan)
            m = [r[f'mad_tta_{c}'] for r in fold_res if f'mad_tta_{c}' in r and not np.isnan(r[f'mad_tta_{c}'])]
            mad_vals.append(np.mean(m) if m else np.nan)

    d_str = f"{np.nanmean(dice_vals):.4f}±{np.nanstd(dice_vals):.4f}"
    i_str = f"{np.nanmean(iou_vals):.4f}±{np.nanstd(iou_vals):.4f}"
    h_str = f"{np.nanmean(hd_vals):.2f}±{np.nanstd(hd_vals):.2f}" if hd_vals else "—"
    m_str = f"{np.nanmean(mad_vals):.2f}±{np.nanstd(mad_vals):.2f}" if mad_vals else "—"
    print(f"  {CLASS_NAMES[c]:<16} {d_str:>12} {i_str:>12} {h_str:>12} {m_str:>12}")

print(f"{'='*70}")

print(f"\n{'─'*70}")
print("PER-VIEW BREAKDOWN (mean foreground Dice, TTA+PP)")
print(f"{'─'*70}")
for view in ['2CH', '4CH']:
    view_res = [r for r in all_results if r['view'] == view]
    _, fg_dice = compute_stats(view_res, 'dice_tta')
    _, fg_iou = compute_stats(view_res, 'iou_tta')
    _, fg_hd = compute_stats(view_res, 'hd_tta')
    print(f"  {view}: Dice={fg_dice:.4f}  IoU={fg_iou:.4f}  HD={fg_hd:.2f}px  (n={len(view_res)})")
print(f"{'='*70}")

print(f"\n{'─'*70}")
print("PER-PHASE BREAKDOWN (mean foreground Dice, TTA+PP)")
print(f"{'─'*70}")
for phase in ['ED', 'ES', 'seq']:
    phase_res = [r for r in all_results if r['phase'] == phase]
    if not phase_res:
        continue
    _, fg_dice = compute_stats(phase_res, 'dice_tta')
    _, fg_iou = compute_stats(phase_res, 'iou_tta')
    _, fg_hd = compute_stats(phase_res, 'hd_tta')
    label = {'ED': 'End-Diastole', 'ES': 'End-Systole', 'seq': 'Half-Sequence'}[phase]
    print(f"  {label}: Dice={fg_dice:.4f}  IoU={fg_iou:.4f}  HD={fg_hd:.2f}px  (n={len(phase_res)})")
print(f"{'='*70}")


# ══════════════════════════════════════════════════════════════
# 3. CONFUSION MATRIX
# ══════════════════════════════════════════════════════════════

all_preds_flat = np.concatenate([r['pred_np'].flatten() for r in all_results])
all_masks_flat = np.concatenate([r['mask_np'].flatten() for r in all_results])

cm = confusion_matrix(all_masks_flat, all_preds_flat, labels=range(NUM_CLASSES))
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

im0 = axes[0].imshow(cm, cmap='Blues')
axes[0].set_title('Confusion Matrix (Pixel Counts)', fontsize=13, fontweight='bold')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        color = 'white' if cm_norm[i, j] > 0.5 else 'black'
        axes[0].text(j, i, f"{cm[i,j]:,.0f}", ha='center', va='center', fontsize=9, color=color)

im1 = axes[1].imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=13, fontweight='bold')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        color = 'white' if cm_norm[i, j] > 0.5 else 'black'
        axes[1].text(j, i, f"{cm_norm[i,j]:.3f}", ha='center', va='center', fontsize=11, color=color)

for ax in axes:
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('Ground Truth', fontsize=11)

plt.colorbar(im0, ax=axes[0], fraction=0.046)
plt.colorbar(im1, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved confusion_matrix.png")


# ══════════════════════════════════════════════════════════════
# 4. PER-FOLD DICE BAR CHART
# ══════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(N_FOLDS)
width = 0.2
colors = ['#2196F3', '#FF9800', '#4CAF50']

for i, c in enumerate(range(1, NUM_CLASSES)):
    vals = []
    for fold in range(N_FOLDS):
        fold_res = [r for r in all_results if r['fold'] == fold]
        d = [r[f'dice_tta_{c}'] for r in fold_res if f'dice_tta_{c}' in r and not np.isnan(r[f'dice_tta_{c}'])]
        vals.append(np.mean(d))
    ax.bar(x + i * width, vals, width, label=CLASS_NAMES[c], color=colors[i])

mean_fg_all = np.mean([fold_stats[f]['dice_tta'] for f in range(N_FOLDS)])
ax.axhline(y=mean_fg_all, color='red', linestyle='--', alpha=0.7, label=f'Mean FG: {mean_fg_all:.4f}')
ax.set_xlabel('Fold', fontsize=12)
ax.set_ylabel('Dice Score', fontsize=12)
ax.set_title('Per-Class Dice Score Across 5 Folds (TTA+PP)', fontsize=13, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels([f'Fold {i}' for i in range(N_FOLDS)])
ax.set_ylim(0.85, 0.96)
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fold_dice_barplot.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved fold_dice_barplot.png")


# ══════════════════════════════════════════════════════════════
# 5. QUALITATIVE RESULTS — BEST PREDICTIONS
# ══════════════════════════════════════════════════════════════

for r in all_results:
    fg_dices = [r[f'dice_tta_{c}'] for c in range(1, NUM_CLASSES) if f'dice_tta_{c}' in r and not np.isnan(r[f'dice_tta_{c}'])]
    r['fg_dice'] = np.mean(fg_dices) if fg_dices else 0

sorted_results = sorted(all_results, key=lambda r: r['fg_dice'], reverse=True)

legend_elements = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor=np.array(SEG_COLORS[1])/255, markersize=12, label='LV Cavity'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor=np.array(SEG_COLORS[2])/255, markersize=12, label='Myocardium'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor=np.array(SEG_COLORS[3])/255, markersize=12, label='Left Atrium'),
]

n_show = 8
best = sorted_results[:n_show]

fig, axes = plt.subplots(n_show, 3, figsize=(10, 3 * n_show))
fig.suptitle(f'Best Predictions — 5-Fold CV (Mean FG Dice: {mean_fg_all:.4f})',
             fontsize=14, fontweight='bold', y=1.01)

for row, r in enumerate(best):
    axes[row, 0].imshow(r['img_np'], cmap='gray')
    axes[row, 0].set_ylabel(f"{r['view']} {r['phase']}\nDice={r['fg_dice']:.3f}", fontsize=9)
    axes[row, 0].set_title('Input' if row == 0 else '', fontsize=11)
    axes[row, 0].set_xticks([])
    axes[row, 0].set_yticks([])
    axes[row, 1].imshow(mask_to_rgb(r['mask_np']))
    axes[row, 1].set_title('Ground Truth' if row == 0 else '', fontsize=11)
    axes[row, 1].axis('off')
    axes[row, 2].imshow(mask_to_rgb(r['pred_np']))
    axes[row, 2].set_title('Prediction' if row == 0 else '', fontsize=11)
    axes[row, 2].axis('off')

fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=11, bbox_to_anchor=(0.5, -0.01))
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'best_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved best_predictions.png")


# ══════════════════════════════════════════════════════════════
# 6. FAILURE CASES — WORST PREDICTIONS
# ══════════════════════════════════════════════════════════════

worst = sorted_results[-n_show:][::-1]

fig, axes = plt.subplots(n_show, 3, figsize=(10, 3 * n_show))
fig.suptitle('Failure Cases — Worst Predictions',
             fontsize=14, fontweight='bold', y=1.01)

for row, r in enumerate(worst):
    axes[row, 0].imshow(r['img_np'], cmap='gray')
    axes[row, 0].set_ylabel(f"{r['view']} {r['phase']}\nDice={r['fg_dice']:.3f}", fontsize=9)
    axes[row, 0].set_title('Input' if row == 0 else '', fontsize=11)
    axes[row, 0].set_xticks([])
    axes[row, 0].set_yticks([])
    axes[row, 1].imshow(mask_to_rgb(r['mask_np']))
    axes[row, 1].set_title('Ground Truth' if row == 0 else '', fontsize=11)
    axes[row, 1].axis('off')
    axes[row, 2].imshow(mask_to_rgb(r['pred_np']))
    axes[row, 2].set_title('Prediction' if row == 0 else '', fontsize=11)
    axes[row, 2].axis('off')

fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=11, bbox_to_anchor=(0.5, -0.01))
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'failure_cases.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved failure_cases.png")


# ══════════════════════════════════════════════════════════════
# 7. OVERLAY VISUALIZATION
# ══════════════════════════════════════════════════════════════

diverse = []
for view in ['2CH', '4CH']:
    for phase in ['ED', 'ES']:
        candidates = [r for r in sorted_results if r['view'] == view and r['phase'] == phase]
        if candidates:
            mid = len(candidates) // 2
            diverse.append(candidates[mid])
            quarter = len(candidates) // 4
            diverse.append(candidates[quarter])

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Segmentation Overlay — Diverse Samples', fontsize=14, fontweight='bold')

for i, r in enumerate(diverse[:8]):
    row, col = i // 4, i % 4
    img = r['img_np']
    pred = r['pred_np']

    overlay = np.stack([img, img, img], axis=-1)
    overlay = (overlay - overlay.min()) / (overlay.max() - overlay.min() + 1e-8)

    for c in range(1, NUM_CLASSES):
        mask_c = (pred == c)
        color = np.array(SEG_COLORS[c]) / 255.0
        for ch in range(3):
            overlay[:, :, ch] = np.where(mask_c, overlay[:, :, ch] * 0.5 + color[ch] * 0.5, overlay[:, :, ch])

    axes[row, col].imshow(overlay)
    axes[row, col].set_title(f"{r['view']} {r['phase']} — {r['fg_dice']:.3f}", fontsize=10)
    axes[row, col].axis('off')

fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=11, bbox_to_anchor=(0.5, -0.01))
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'overlay_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved overlay_results.png")


# ══════════════════════════════════════════════════════════════
# 8. SAVE ALL RESULTS TO JSON
# ══════════════════════════════════════════════════════════════

json_results = {
    'model': 'DeepSupSMPUNet (ResNet34 + SCSE + Deep Supervision, 24.5M params)',
    'dataset': 'CAMUS (500 patients, 5-fold patient-level CV)',
    'loss': 'DiceCE (CE=0.3, Dice=0.7)',
    'per_fold': {},
    'overall': {
        'dice_raw': f"{np.mean([fold_stats[f]['dice_raw'] for f in range(N_FOLDS)]):.4f} ± {np.std([fold_stats[f]['dice_raw'] for f in range(N_FOLDS)]):.4f}",
        'dice_tta': f"{np.mean([fold_stats[f]['dice_tta'] for f in range(N_FOLDS)]):.4f} ± {np.std([fold_stats[f]['dice_tta'] for f in range(N_FOLDS)]):.4f}",
        'iou': f"{np.mean([fold_stats[f]['iou'] for f in range(N_FOLDS)]):.4f} ± {np.std([fold_stats[f]['iou'] for f in range(N_FOLDS)]):.4f}",
        'hd_px': f"{np.mean([fold_stats[f]['hd'] for f in range(N_FOLDS)]):.2f} ± {np.std([fold_stats[f]['hd'] for f in range(N_FOLDS)]):.2f}",
        'mad_px': f"{np.mean([fold_stats[f]['mad'] for f in range(N_FOLDS)]):.2f} ± {np.std([fold_stats[f]['mad'] for f in range(N_FOLDS)]):.2f}",
    },
    'per_view': {},
    'per_phase': {},
}

for fold in range(N_FOLDS):
    json_results['per_fold'][f'fold_{fold}'] = fold_stats[fold]

for view in ['2CH', '4CH']:
    view_res = [r for r in all_results if r['view'] == view]
    _, fg = compute_stats(view_res, 'dice_tta')
    json_results['per_view'][view] = float(fg)

for phase in ['ED', 'ES', 'seq']:
    phase_res = [r for r in all_results if r['phase'] == phase]
    if phase_res:
        _, fg = compute_stats(phase_res, 'dice_tta')
        json_results['per_phase'][phase] = float(fg)

results_path = os.path.join(SAVE_DIR, 'cv5_full_results.json')
with open(results_path, 'w') as f:
    json.dump(json_results, f, indent=2, default=float)
print(f"\nAll results saved to {results_path}")


# ══════════════════════════════════════════════════════════════
# 9. FINAL SUMMARY
# ══════════════════════════════════════════════════════════════

print(f"\n\n{'═'*70}")
print("FINAL PROJECT SUMMARY")
print(f"{'═'*70}")
print(f"  Model:      ResNet34 + SCSE + Deep Supervision (24.5M params)")
print(f"  Dataset:    CAMUS (500 patients, 5-fold patient-level CV)")
print(f"  Loss:       DiceCE (CE=0.3, Dice=0.7)")
print(f"{'─'*70}")

dice_vals = [fold_stats[f]['dice_tta'] for f in range(N_FOLDS)]
iou_vals = [fold_stats[f]['iou'] for f in range(N_FOLDS)]
hd_vals = [fold_stats[f]['hd'] for f in range(N_FOLDS)]
mad_vals = [fold_stats[f]['mad'] for f in range(N_FOLDS)]

print(f"  Dice (TTA):  {np.mean(dice_vals):.4f} ± {np.std(dice_vals):.4f}")
print(f"  IoU (TTA):   {np.mean(iou_vals):.4f} ± {np.std(iou_vals):.4f}")
print(f"  HD (px):     {np.mean(hd_vals):.2f} ± {np.std(hd_vals):.2f}")
print(f"  MAD (px):    {np.mean(mad_vals):.2f} ± {np.std(mad_vals):.2f}")
print(f"{'─'*70}")

print(f"\n  Literature comparison (CAMUS, mean FG Dice):")
print(f"  {'Method':<35} {'Dice':>8}")
print(f"  {'─'*45}")
print(f"  {'nnU-Net (lightweight ablation)':<35} {'~0.893':>8}")
print(f"  {'UwU-Net (263K params)':<35} {'~0.890':>8}")
print(f"  {'Ours (5-fold CV + TTA)':<35} {np.mean(dice_vals):>8.4f}")
print(f"  {'TC-SegNet (ASPP+SE)':<35} {'0.928':>8}")
print(f"{'═'*70}")

print(f"\nFigures saved to {SAVE_DIR}:")
print(f"  - confusion_matrix.png")
print(f"  - fold_dice_barplot.png")
print(f"  - best_predictions.png")
print(f"  - failure_cases.png")
print(f"  - overlay_results.png")
print(f"  - cv5_full_results.json")